Paquetes necesarios

In [ ]:
import cv2  
import math 
import csv
from ultralytics import YOLO
from paddleocr import PaddleOCR
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]

#Para un vídeo 
filename = "assets/C0142.mp4"
results = model(source=filename,
                show=True,
                imgsz=320)

cv2.destroyAllWindows()



image 1/1 c:\Users\skyri\OneDrive\Documentos\4 AÑO I.inforatica\VC\VC-Practicas\P4\assets\test.jpg: 192x320 1 person, 3 cars, 2 motorcycles, 26.9ms
Speed: 0.7ms preprocess, 26.9ms inference, 1.5ms postprocess per image at shape (1, 3, 192, 320)


In [ ]:
# Modelo RTDETR

### Codigo basico sin OCR

In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores
model2 = YOLO('yolo11nano.pt') # Matrículas

clases_vehiculos = [2, 3, 5, 7]

cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),  # person
    2: set(),  # coches
    3: set(),  # motos
    5: set(),  # trucks
    7: set()   # buses
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])


while True:
    ret, frame = cap.read()
    frame_count += 1
    
    if not ret:
        break

    # Para cada detección
    results = model.track(source=frame, persist=True, imgsz=320, verbose=False)
    
    for box in results[0].boxes:
        # Contenedor
        x1, y1, x2, y2 = box.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 

        # Clase
        cls = int(box.cls[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # color verde para el contenedor
        cv2.putText(frame, model.names[cls] , (x1, max(15, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    
        # Parametros Csv
        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1  # Valor por defecto si no hay ID de seguimiento
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)


        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos:
            vehiculo_recortado = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
            
            results2 = model2(source=vehiculo_recortado, imgsz=320)

            if len(results2[0].boxes) > 0:
                for box2 in results2[0].boxes:
                    print(f"El modelo 2 ha detectado: {model2.names[int(box2.cls[0])]}")

                    if int(box2.cls[0]) == 0:  # Si la clase detectada es matrícula
                        tiene_mat = 1
                        conf_mat = float(box2.conf[0])

                        x1_2, y1_2, x2_2, y2_2 = box2.xyxy[0]
                        x1_2, y1_2, x2_2, y2_2 = int(x1_2), int(y1_2), int(x2_2), int(y2_2)

                        g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                        g_x2, g_y2 = x1 + x2_2, y1 + y2_2

                        cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2) # color azul para la matrícula

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(20) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls}: {len(boxes)} detecciones")

cv2.destroyAllWindows()
output_video.release()
cap.release()
csv_file.close()


0: 288x320 1 bus, 30.5ms
Speed: 1.2ms preprocess, 30.5ms inference, 0.9ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: bus

0: 288x320 1 bus, 36.1ms
Speed: 1.5ms preprocess, 36.1ms inference, 0.9ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: bus

0: 288x320 1 bus, 44.4ms
Speed: 2.6ms preprocess, 44.4ms inference, 1.6ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: bus

0: 288x320 1 bus, 35.2ms
Speed: 1.4ms preprocess, 35.2ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: bus

0: 288x320 1 bus, 49.1ms
Speed: 1.3ms preprocess, 49.1ms inference, 1.5ms postprocess per image at shape (1, 3, 288, 320)
El modelo 2 ha detectado: bus

0: 256x320 1 bus, 35.1ms
Speed: 1.4ms preprocess, 35.1ms inference, 0.8ms postprocess per image at shape (1, 3, 256, 320)
El modelo 2 ha detectado: bus

0: 256x320 1 person, 1 bus, 45.6ms
Speed: 1.1ms preprocess, 45.6ms inference, 1.3

#### Codigo con PaddleOCR

In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores
model2 = YOLO('yolo11nano.pt') # Matrículas
ocr = PaddleOCR(use_angle_cls=True, lang='es')

clases_vehiculos = [2, 3, 5, 7]

cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),  # person
    2: set(),  # coches
    3: set(),  # motos
    5: set(),  # trucks
    7: set()   # buses
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])


while True:
    ret, frame = cap.read()
    frame_count += 1
    
    if not ret:
        break

    # Para cada detección
    results = model.track(source=frame, persist=True, imgsz=320, verbose=False)
    
    for box in results[0].boxes:
        # Contenedor
        x1, y1, x2, y2 = box.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 

        # Clase
        cls = int(box.cls[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # color verde para el contenedor
        cv2.putText(frame, model.names[cls] , (x1, max(15, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        

        # Parametros Csv
        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1  # Valor por defecto si no hay ID de seguimiento
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)


        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos:
            vehiculo_recortado = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
            
            results2 = model2(source=vehiculo_recortado, imgsz=320)

            if len(results2[0].boxes) > 0:
                for box2 in results2[0].boxes:
                    print(f"El modelo 2 ha detectado: {model2.names[int(box2.cls[0])]}")

                    if int(box2.cls[0]) == 0:  # Si la clase detectada es matrícula
                        tiene_mat = 1
                        conf_mat = float(box2.conf[0])

                        x1_2, y1_2, x2_2, y2_2 = box2.xyxy[0]
                        x1_2, y1_2, x2_2, y2_2 = int(x1_2), int(y1_2), int(x2_2), int(y2_2)

                        g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                        g_x2, g_y2 = x1 + x2_2, y1 + y2_2

                        matricula_recortada = frame[max(0, g_y1):max(0, g_y2), max(0, g_x1):max(0, g_x2)]
                        
                        if matricula_recortada.size > 0:
                            result = ocr.predict(matricula_recortada)[0]

                            if len(result['rec_texts']) > 0:
                                texto_matricula = result['rec_texts'][0]

                        cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2) # color azul para la matrícula

                        if texto_matricula != "":
                            cv2.putText(frame, texto_matricula, (g_x1, max(15, g_y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(20) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls}: {len(boxes)} detecciones")

cv2.destroyAllWindows()
output_video.release()
cap.release()

#### Codigo con VLMs

In [ ]:
# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') # Contenedores
model2 = YOLO('yolo11nano.pt') # Matrículas

Device = "cpu"  # or "cpu"

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model_ocr = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                dtype=torch.bfloat16,
                                                _attn_implementation="flash_attention_2" if Device == "cuda" else "eager").to(Device)

clases_vehiculos = [2, 3, 5, 7]

cap = cv2.VideoCapture("assets/C0142.mp4")

total_clases = {
    0: set(),  # person
    2: set(),  # coches
    3: set(),  # motos
    5: set(),  # trucks
    7: set()   # buses
}

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

output_video = cv2.VideoWriter('output_video.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

frame_count = 0

csv_file = open('detections.csv', mode='w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['frame', 'class', 'confidence', 'track_id', 'x1', 'y1', 'x2', 'y2', 'matrícula_en_su_caso', 'confianza', 'mx1', 'my1', 'mx2', 'my2', 'texto_matricula'])


while True:
    ret, frame = cap.read()
    frame_count += 1
    
    if not ret:
        break

    # Para cada detección
    results = model.track(source=frame, persist=True, imgsz=320, verbose=False)
    
    for box in results[0].boxes:
        # Contenedor
        x1, y1, x2, y2 = box.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) 


        # Clase
        cls = int(box.cls[0])
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # color verde para el contenedor
        cv2.putText(frame, model.names[cls] , (x1, max(15, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

       

        # Parametros Csv
        conf_model = float(box.conf[0])
        objeto = model.names[cls]

        track_id = -1  # Valor por defecto si no hay ID de seguimiento
        if box.id is not None:
            track_id = int(box.id[0])

            if cls in total_clases:
                total_clases[cls].add(track_id)


        tiene_mat = 0
        conf_mat = 0.0
        g_x1, g_y1, g_x2, g_y2 = 0, 0, 0, 0
        texto_matricula = ""

        if cls in clases_vehiculos:
            vehiculo_recortado = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
            
            results2 = model2(source=vehiculo_recortado, imgsz=320)

            if len(results2[0].boxes) > 0:
                for box2 in results2[0].boxes:
                    print(f"El modelo 2 ha detectado: {model2.names[int(box2.cls[0])]}")

                    if int(box2.cls[0]) == 0:  # Si la clase detectada es matrícula
                        tiene_mat = 1
                        conf_mat = float(box2.conf[0])

                        x1_2, y1_2, x2_2, y2_2 = box2.xyxy[0]
                        x1_2, y1_2, x2_2, y2_2 = int(x1_2), int(y1_2), int(x2_2), int(y2_2)

                        g_x1, g_y1 = x1 + x1_2, y1 + y1_2
                        g_x2, g_y2 = x1 + x2_2, y1 + y2_2
                        
                        matricula_recortada = frame[max(0, g_y1):max(0, g_y2), max(0, g_x1):max(0, g_x2)]
                        
                        if matricula_recortada.size > 0:
                            matricula_rgb = cv2.cvtColor(matricula_recortada, cv2.COLOR_BGR2RGB)
                            image_pil = Image.fromarray(matricula_rgb)

                            messages = [
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "image"},
                                        {"type": "text", "text": "Can you read the text in this image? Return ONLY the letters and numbers, nothing else."}
                                    ]
                                }
                            ]

                            prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
                            inputs = processor(text=prompt, images=[image_pil], return_tensors="pt").to(Device)

                            generated_ids = model_ocr.generate(**inputs, max_new_tokens=10)
                            generated_texts = processor.batch_decode(
                                generated_ids,
                                skip_special_tokens=True,
                            )

                            if "Assistant" in generated_texts:
                                texto_matricula = generated_texts.split("Assistant:")[-1].strip()
                                                


                        cv2.rectangle(frame, (g_x1, g_y1), (g_x2, g_y2), (255, 0, 0), 2) # color azul para la matrícula

                        if texto_matricula != "":
                            cv2.putText(frame, texto_matricula, (g_x1, max(15, g_y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        writer.writerow([
            frame_count, objeto, round(conf_model, 2), track_id,
            x1, y1, x2, y2,
            tiene_mat, round(conf_mat, 2), g_x1, g_y1, g_x2, g_y2, texto_matricula
        ])
    output_video.write(frame)

    cv2.imshow("Detección de vehículos", frame)
    if cv2.waitKey(20) == 27:
        break

for cls, boxes in total_clases.items():
    print(f"Clase {cls}: {len(boxes)} detecciones")

cv2.destroyAllWindows()
output_video.release()
cap.release()